## 授權與來源（License & Attribution）

本課程自編與改編內容：Copyright 2026 leonjyentub；授權：Apache License 2.0（`../LICENSE`）。
完整來源及授權資訊：`../SOURCES.md`、`../THIRD_PARTY_NOTICES.md`。

本篇為課程環境導覽，無宣稱直接複製 handson-mlp 程式。

本授權只適用於本 Notebook 中權利人有權授權的程式與文字；第三方資料、圖片與其他權利依各原始條款。

# 00｜課程導覽

環境、資料與重現性檢查。對應 `slides/00_課程導覽.md`。

**Colab 示範**：直接上傳此 `.ipynb`，從上到下執行；第一個程式格會安裝缺少的套件，所需資料在使用時自動下載。NumPy 與 scikit-learn 實驗使用 CPU；這些程式沒有可直接切換的 CUDA 後端。輸出存於目前工作目錄。


In [ ]:
# 獨立執行：Colab 上傳此檔後，從這一格依序執行；無需掛載 Git 專案。
import importlib.util, subprocess, sys
_needed = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3", "matplotlib": "matplotlib>=3.7,<4", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2", "cloudpickle": "cloudpickle>=3,<4", "ipywidgets": "ipywidgets>=8,<9"}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib, io, json, os, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# 本機在 programs/ 執行時沿用 data/、outputs/；Colab 則使用自己的工作目錄。
DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
SEED = 42
FAST = os.environ.get("MLCOURSE_PROFILE", "fast") != "full"
_SOURCE_COMMIT = "9e29abbbea6a3ff7250dea64c43c1d6c723095d7"
_SOURCES = {
    "housing.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/housing.tgz", "2364609dc48bec7df3ba9dbb7041478e704ecddcee70ef1827ec3fc49d22c0cc"),
    "lifesat.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/lifesat/lifesat.csv", "e247f7f6c19b0d8c55fb2af293f4fb3b710e31476534ad8e9b7e06ea546a2099"),
}

def setup():
    plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})
    OUT.mkdir(parents=True, exist_ok=True)
    return {"profile": "fast" if FAST else "full", "seed": SEED, "data": str(DATA)}

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

def data_path(name):
    if name not in (*_SOURCES, "mnist_784_v1.npz"):
        raise ValueError(f"未知資料檔：{name}")
    DATA.mkdir(parents=True, exist_ok=True)
    path = DATA / name
    if name in _SOURCES:
        url, expected_sha = _SOURCES[name]
        if not path.exists():
            print(f"下載 {name} …")
            with urllib.request.urlopen(url, timeout=180) as response:
                blob = response.read()
            if name == "housing.csv":
                with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
                    blob = archive.extractfile("housing/housing.csv").read()
            if hashlib.sha256(blob).hexdigest() != expected_sha:
                raise ValueError(f"資料校驗失敗：{name}")
            path.write_bytes(blob)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError(f"本機資料校驗失敗：{name}")
        info = {"url": url, "sha256": expected_sha}
    else:
        if not path.exists():
            print("從 OpenML 下載 MNIST（首次約需數分鐘）…")
            from sklearn.datasets import fetch_openml
            ds = fetch_openml("mnist_784", version=1, as_frame=False,
                              data_home=DATA / "openml_cache", parser="auto")
            X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
            assert X.shape == (70000, 784) and y.shape == (70000,)
            np.savez_compressed(path, X=X, y=y)
        with np.load(path) as archive:
            assert archive["X"].shape == (70000, 784)
            assert archive["y"].shape == (70000,)
        info = {"url": "https://www.openml.org/d/554",
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    manifest_file = DATA / "manifest.json"
    manifest = json.loads(manifest_file.read_text()) if manifest_file.exists() else {}
    manifest[name] = info
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    return path

def housing_split():
    from sklearn.model_selection import train_test_split
    frame = pd.read_csv(data_path("housing.csv"))
    income = pd.cut(frame.median_income, [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
    train, test = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=income)
    assert not set(train.index) & set(test.index)
    return train, test

setup()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. 核對真正執行的 Python 與套件
`uv.lock` 固定安裝結果；Notebook 的 Kernel 也必須使用同一個環境。

In [ ]:
import sys, platform, importlib.metadata as meta
print(sys.executable)
print(platform.platform())
versions = {p: meta.version(p) for p in ['numpy','pandas','scipy','scikit-learn','matplotlib']}
pd.Series(versions, name='installed version')

## 2. 核對教材資料
住房資料為歷史 California Housing 區域資料，非目前房價；MNIST 為 OpenML ID 554 / version 1，固定 70,000 張。資料 hash 不同時先追查來源，不直接忽略檢查。

In [ ]:
import json
for name in ('housing.csv', 'lifesat.csv', 'mnist_784_v1.npz'):
    data_path(name)
manifest = json.loads((DATA/'manifest.json').read_text())
for name in manifest:
    p = data_path(name)
    print(name, f'{p.stat().st_size/1024**2:.2f} MiB', manifest[name]['sha256'][:16])
assert set(manifest) == {'housing.csv','lifesat.csv','mnist_784_v1.npz'}
report('environment', {'python':sys.version, 'versions':versions, 'profile':'fast' if FAST else 'full'})

## 3. 同一個 seed 能保證什麼？
在相同程式、資料順序與環境下重現亂數；它不保證新資料加入後仍有相同切分，也不消除取樣不確定性。

In [ ]:
a = np.random.default_rng(SEED).normal(size=6)
b = np.random.default_rng(SEED).normal(size=6)
assert np.array_equal(a,b)
print(a)
plt.plot(a, 'o-', label='same seed, same draws')
plt.xlabel('Draw index'); plt.ylabel('Value'); plt.legend()
savefig('00_environment_smoke')

## 操作與課前檢查

1. `Kernel → Restart Kernel and Run All Cells`，確認沒有錯誤。
2. 預設 **fast**：Housing 最多 6,000 筆開發資料、MNIST 12,000 筆開發資料；測試保留集合不縮減。
3. 完整模式在啟動 Jupyter **之前** 設定：`MLCOURSE_PROFILE=full uv run jupyter lab`。需重啟核心；完整模式運算較久，本次驗證使用 fast。
4. 圖表另存 `outputs/figures/` 的 PNG 與可編輯 SVG；評估數據另存 `outputs/reports/`。
5. 無網路時可閱讀已執行輸出，或使用已附資料快取重新運算。套件第一次安裝仍需網路或事先準備 uv 快取。

**自我檢核：**`sys.executable` 應指向哪裡？同一 seed 換了資料順序，是否能保證相同測試樣本？

<details><summary>參考答案</summary>應指向 programs/.venv/bin/python。重新排列資料後，依位置隨機切分通常改變；需要不可變 ID 的穩定規則才能維持成員。</details>